In [ ]:
from pathlib import Path
from metasmith.python_api import Agent, Source, Std, DataInstanceLibrary, TransformInstanceLibrary, WorkflowTask
from metasmith.python_api import DataTypeLibrary, Endpoint
from local.constants import WORKSPACE_ROOT

# dtypes, containers, transforms = Std()

path_to_agent_home = Path("./cache/local_home").resolve()
smith = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
# smith.Deploy()

In [ ]:
mock_types = DataTypeLibrary(types=dict(
    a=Endpoint({"test", "a"}),
    b=Endpoint({"test", "b"}),
    c=Endpoint({"test", "c"}),
))

transforms = TransformInstanceLibrary("./transforms/simple_1", include_std=False)
transforms.AddTypeLibrary("mock", mock_types)
transforms.AddStub("no_op")
transforms.Save()

In [ ]:
inputs = DataInstanceLibrary("./cache/dev21.mock.xgdb")
samples = []
for i in range(10):
    in_path = WORKSPACE_ROOT/f"main/local_mock/cache/test/mock_d.{i}"
    in_path.parent.mkdir(exist_ok=True)
    with open(in_path, "w") as f:
        f.write("10")
    inputs.AddTypeLibrary("mock", mock_types)
    inputs.AddItem(in_path, "mock::a")
    samples.append(in_path)
inputs.Save()
for p, n, e in inputs.Iterate():
    print(n, e, e.parents)

In [ ]:
for loc, t,  in transforms.IterateTransforms():
    print(t.model)

In [ ]:
task = smith.GenerateWorkflow(
    samples    = [inputs.AsView({p}) for p in samples],
    resources  = [],
    transforms = [transforms],
    targets    = [mock_types["b"]]
)
with open(WORKSPACE_ROOT/"secrets/slurm_account_fir") as f:
    SLURM_ACCOUNT = f.read()

task.config = dict(
    nextflow = dict(
        preset="slurm",
        slurm_account=SLURM_ACCOUNT,
        cpus=1,
        array=10,
        queueSize=500,
        memory=8,
        time=3,
    )
)
print(task.GetKey())

In [ ]:
smith.StageWorkflow(task, on_exist='clear', verify_external_paths=False)

In [ ]:
smith.RunWorkflow(task)

In [ ]:
assert False

In [ ]:
import pandas as pd
import json
from datetime import datetime

with open("./cache/nxf_report.dev21.html") as f:
    found = False
    for l in f:
        if l.strip().startswith('window.data = { "trace":['): 
            found = True
            continue
        if not found: 
            continue

        d = json.loads('{ "trace":[' + l[:-2])
        break
dfo = pd.DataFrame(d['trace'])
df = dfo
df = df[df.attempt == "3"]
df = df[~df.status.isin({"COMPLETED", "ABORTED"})]

df = df['hash, status, exit, submit, peak_rss, peak_vmem, duration, cpu_model, error_action, attempt'.split(', ')]
start = datetime.fromtimestamp(0)
df['duration'] = df.duration.apply(lambda x: (datetime.fromtimestamp(int(x)/1000.0)-start))
df

In [ ]:
df.columns

In [ ]:
dfo.time.